[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashkanhszd/m8_data_science_module/blob/main/M8_8.31_33_ashkan_python_advanced/M8_8.31_33_Funciones_automatizacion_reproducibilidad.ipynb)

# M8_8.31–M8_8.33 · Funciones, automatización y flujos reproducibles

**Máster en Hidrogeología y Modelación (UB) · Módulo 8: Herramientas y Data Science**

### Objetivos de la tarde

1. Convertir el control de calidad y los cálculos de sesiones anteriores en **funciones** reutilizables, documentadas y probadas.
2. Guardar las funciones en un **módulo** (`.py`), importarlas y ejecutar un **script**.
3. **Automatizar** un análisis para muchos pozos, con configuración separada del código, control de errores y registro.
4. Construir un **pequeño proyecto reproducible** y comprobar que funciona con **datos nuevos**.

### Plan de la tarde

| Sesión | Contenido | Tiempo aprox. |
|---|---|---|
| **M8_8.31** | Preparación de datos con funciones: responsabilidad única, docstrings, validación, pruebas, módulos | 55 min |
| *Descanso* | | 10 min |
| **M8_8.32** | Automatización: configuración, bucles sobre pozos, nombres trazables, errores, registro, script | 50 min |
| *Descanso* | | 10 min |
| **M8_8.33** | Taller: proyecto reproducible con datos nuevos (campaña 2026) · README · ejecución desde cero · interpretación | 60 min |

Iconos: 💡 idea clave · 📖 documentación · ✏️ ejercicio · ✅ comprobación · ⚠️ error frecuente

## 0. Preparación

In [ ]:
# ==================================================================
# PREPARACIÓN DEL ENTORNO · ejecuta siempre esta celda en primer lugar
# ==================================================================
# · En Google Colab: descarga (clona) el repositorio del curso y se sitúa en él.
# · En un ordenador propio: busca la carpeta del repositorio.
# No hace falta entender cada línea todavía; la estructura se explica en M8_8.20.

import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/Ashkanhszd/m8_data_science_module.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    ROOT = Path("/content/m8_data_science_module")
    if not ROOT.exists():
        print("Descargando el repositorio del curso...")
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(ROOT)], check=True)
else:
    ROOT = None
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "data" / "input").exists():
            ROOT = carpeta
            break
    if ROOT is None:
        raise FileNotFoundError("No se encuentra data/input: abre el notebook desde la carpeta del repositorio.")

os.chdir(ROOT)
INPUT = ROOT / "data" / "input"          # datos originales (no se modifican)
DATABASE = ROOT / "data" / "database"    # base de datos SQLite
OUTPUT = ROOT / "data" / "output"        # resultados que generamos
for subcarpeta in ["tables", "figures", "logs"]:
    (OUTPUT / subcarpeta).mkdir(parents=True, exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "ordenador local")
print("Carpeta del repositorio:", ROOT)
print("Python", sys.version.split()[0])

In [ ]:
import importlib
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("NumPy", np.__version__, "| pandas", pd.__version__, "| Matplotlib", matplotlib.__version__)

pozos = pd.read_csv(INPUT / "pozos.csv")

---
# M8_8.31 · Preparación de datos con funciones

## 1. Del código exploratorio al código reutilizable

En M8_8.27 limpiamos las mediciones con una celda larga de instrucciones. Funcionó, pero ¿qué pasa cuando llegan datos nuevos (y llegarán: en M8_8.23 recibimos la campaña de 2026)? Habría que **copiar y pegar** la celda y modificarla, con el riesgo de olvidar un paso o cambiar algo sin querer.

| Código exploratorio | Código reutilizable |
|---|---|
| sirve para probar ideas | sirve para repetir un procedimiento |
| variables sueltas y valores escritos dentro del código | **funciones** con parámetros |
| se ejecuta una vez | se ejecuta muchas veces, con datos distintos |
| se entiende mientras lo escribes | se entiende meses después, gracias a la **documentación** |

Una **función** encapsula un paso del procedimiento. Es la misma idea que un algoritmo dentro de un modelo de Graphical Modeler.

## 2. Qué hace buena a una función

1. **Una sola responsabilidad**, que se pueda describir en una frase con un verbo: *cargar las mediciones*, *eliminar duplicados*, *calcular la cota*.
2. **Todo lo que necesita entra por los parámetros**, y todo lo que produce sale con `return`. No depende de variables definidas fuera.
3. **No modifica sus entradas**: trabaja sobre una copia (`.copy()`).
4. **Valida** lo que recibe y, si algo va mal, da un **mensaje de error claro**.
5. Tiene una **docstring** que explica qué hace, qué recibe, qué devuelve y qué errores puede lanzar.

## 3. Una función de carga con validación

La docstring sigue el **estilo NumPy**, el mismo que usan NumPy, pandas y la mayoría de paquetes científicos. Si aprendes a escribirla, también aprenderás a leer la documentación de esos paquetes.

In [ ]:
COLUMNAS_MEDICIONES = ["id_pozo", "fecha", "profundidad_nivel_m", "conductividad_uScm"]


def cargar_mediciones(ruta):
    """Lee un CSV de mediciones y comprueba que tiene las columnas obligatorias.

    Parameters
    ----------
    ruta : str o Path
        Ruta al archivo CSV.

    Returns
    -------
    pandas.DataFrame
        Mediciones con la columna ``fecha`` convertida a fecha.

    Raises
    ------
    FileNotFoundError
        Si el archivo no existe.
    ValueError
        Si falta alguna columna obligatoria.
    """
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    datos = pd.read_csv(ruta, parse_dates=["fecha"])

    faltan = set(COLUMNAS_MEDICIONES) - set(datos.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(faltan)}")
    return datos


mediciones = cargar_mediciones(INPUT / "mediciones.csv")
mediciones.shape

In [ ]:
help(cargar_mediciones)

Si la entrada no es válida, la función se detiene con un mensaje que explica el problema. Es mucho mejor fallar pronto y con claridad que producir un resultado incorrecto sin avisar.

In [ ]:
for ruta in [INPUT / "no_existe.csv", INPUT / "pozos.csv"]:
    try:
        cargar_mediciones(ruta)
    except (FileNotFoundError, ValueError) as error:
        print(type(error).__name__, "→", error)

## 4. Funciones pequeñas para cada regla de limpieza

Cada regla de M8_8.27 se convierte en una función. Todas siguen el mismo patrón: reciben un DataFrame, devuelven **una copia modificada** y el **número de filas afectadas**, que usaremos para el registro.

In [ ]:
def eliminar_duplicados_exactos(datos):
    """Devuelve una copia sin filas repetidas y el número de filas eliminadas."""
    limpios = datos.drop_duplicates().copy()
    return limpios, len(datos) - len(limpios)


def corregir_identificadores(datos, correcciones):
    """Sustituye identificadores de pozo mal escritos.

    Parameters
    ----------
    datos : pandas.DataFrame
        Mediciones con la columna ``id_pozo``.
    correcciones : dict
        Pares {"incorrecto": "correcto"}, por ejemplo {"P1": "P01"}.

    Returns
    -------
    tuple (pandas.DataFrame, int)
        Copia corregida y número de filas modificadas.
    """
    resultado = datos.copy()
    a_corregir = resultado["id_pozo"].isin(list(correcciones))
    resultado["id_pozo"] = resultado["id_pozo"].replace(correcciones)
    return resultado, int(a_corregir.sum())

In [ ]:
paso1, n_dup = eliminar_duplicados_exactos(mediciones)
paso2, n_id = corregir_identificadores(paso1, {"P1": "P01"})

print("Duplicados eliminados:", n_dup, "| Identificadores corregidos:", n_id)
print("El original no cambia:", len(mediciones), "filas y todavía contiene 'P1':", "P1" in set(mediciones["id_pozo"]))

> 💡 **Parámetros en lugar de valores fijos.** `corregir_identificadores` no sabe nada de `P1`: recibe las correcciones como parámetro. La misma función servirá para cualquier otro error de identificador que aparezca en el futuro. La **decisión** (que `P1` es `P01`) queda fuera de la función, donde es visible y se puede documentar.

## 5. Funciones puras y efectos secundarios

Una función **pura** solo calcula: su resultado depende únicamente de sus entradas y no cambia nada fuera de ella. Es fácil de entender y de probar.

Un **efecto secundario** es cualquier cambio fuera de la función: guardar un archivo, modificar un DataFrame recibido, dibujar una figura, escribir un registro. No son malos (¡los necesitamos!), pero conviene **separarlos** de los cálculos: primero se calcula con funciones puras y, al final, se guarda.

> ⚠️ **Error frecuente:** modificar el DataFrame recibido sin hacer `.copy()`. El original cambia "por sorpresa" y, si ejecutas la celda dos veces, obtienes resultados distintos.

In [ ]:
def anadir_cota_piezometrica(datos, pozos):
    """Devuelve una copia con acuífero, cota del terreno y cota piezométrica (h = z - d)."""
    resultado = datos.merge(pozos[["id_pozo", "acuifero", "cota_terreno_m"]],
                            on="id_pozo", how="left", validate="many_to_one")
    resultado["cota_piezometrica_m"] = resultado["cota_terreno_m"] - resultado["profundidad_nivel_m"]
    return resultado

In [ ]:
con_cota = anadir_cota_piezometrica(paso2, pozos)
print("Columnas originales:", list(paso2.columns))
print("Columnas nuevas:    ", list(con_cota.columns))

## 6. Probar las funciones con `assert`

Una **prueba** comprueba que una función hace lo que debe con un caso cuyo resultado **conocemos de antemano**. Lo ideal es usar datos pequeños, inventados para la prueba, en los que el resultado se puede calcular de cabeza.

`assert condición, "mensaje"` no hace nada si la condición es verdadera, y detiene la ejecución con el mensaje si es falsa.

In [ ]:
# Datos de prueba: 3 filas, una de ellas duplicada y un identificador mal escrito
prueba = pd.DataFrame({
    "id_pozo": ["P01", "P01", "P1"],
    "fecha": pd.to_datetime(["2024-01-01", "2024-01-01", "2024-02-01"]),
    "profundidad_nivel_m": [8.0, 8.0, 8.3],
    "conductividad_uScm": [950.0, 950.0, 960.0],
})

resultado, n = eliminar_duplicados_exactos(prueba)
assert len(resultado) == 2 and n == 1, "Debería eliminar exactamente una fila"
assert len(prueba) == 3, "¡La función ha modificado su entrada!"

resultado, n = corregir_identificadores(prueba, {"P1": "P01"})
assert set(resultado["id_pozo"]) == {"P01"} and n == 1

pozos_prueba = pd.DataFrame({"id_pozo": ["P01"], "acuifero": ["Aluvial"], "cota_terreno_m": [100.0]})
h = anadir_cota_piezometrica(resultado, pozos_prueba)["cota_piezometrica_m"]
assert np.allclose(h, [92.0, 92.0, 91.7])

print("✔ Todas las pruebas superadas")

### ✏️ Ejercicio 1 · La función `anular_imposibles`

Escribe una función que:

- reciba las mediciones (`datos`) y la tabla `pozos`;
- ponga a `NaN` las profundidades **negativas** o **mayores que la profundidad del pozo** (`profundidad_pozo_m`);
- **no modifique** `datos` y devuelva la copia **sin** la columna auxiliar `profundidad_pozo_m`;
- devuelva también cuántos valores ha anulado;
- tenga una docstring de estilo NumPy.

Pista: une las mediciones con `pozos[["id_pozo", "profundidad_pozo_m"]]` con `merge`. Esa unión ya crea una tabla nueva.

In [ ]:
def anular_imposibles(datos, pozos):
    """...
    """
    ...
    return resultado, ...


limpios_prueba, n_anulados = anular_imposibles(paso2, pozos)
print("Valores anulados:", n_anulados)

In [ ]:
# ✅ Comprobación
assert n_anulados == 1, "En los datos reales solo hay un valor imposible (P11, 2022-08)"
assert "profundidad_pozo_m" not in limpios_prueba.columns, "Elimina la columna auxiliar antes de devolver"
assert limpios_prueba["profundidad_nivel_m"].max() < 80
assert paso2["profundidad_nivel_m"].max() > 200, "La función no debe modificar su entrada"
assert anular_imposibles.__doc__ and len(anular_imposibles.__doc__) > 30, "Escribe una docstring"
print("✔ Correcto")

## 7. De funciones sueltas a un módulo

Las funciones definidas en un notebook solo existen **en ese notebook** y mientras la sesión está abierta. Para reutilizarlas en otros notebooks, en scripts o en otros proyectos, se guardan en un archivo `.py`: un **módulo**.

Vamos a crear la estructura de un pequeño proyecto:

```text
mi_proyecto/
├── hidro.py            ← módulo con todas las funciones
├── ejecutar_flujo.py   ← script que ejecuta el análisis completo   (M8_8.32)
├── config.json         ← parámetros del análisis                    (M8_8.32)
├── README.md           ← instrucciones                              (M8_8.33)
└── resultados/         ← tablas, figuras y registros generados
```

La instrucción mágica `%%writefile ruta` (solo en notebooks), escrita en la primera línea de una celda, **guarda el contenido de la celda en un archivo** en lugar de ejecutarlo. El módulo contiene las funciones que ya hemos probado y algunas más, que seguimos el mismo patrón: el resto de reglas de limpieza, el resumen por pozo, las figuras, el registro y el flujo completo. **Léelo con calma**: ya entiendes casi todo.

In [ ]:
(ROOT / "mi_proyecto").mkdir(exist_ok=True)

In [ ]:
%%writefile mi_proyecto/hidro.py
"""
hidro.py · Funciones reutilizables para el análisis de datos hidrogeológicos del Módulo 8.

Creado en M8_8.31. Cada función tiene una única responsabilidad, recibe sus
entradas como parámetros y devuelve su resultado (no depende de variables externas).
"""

import datetime
import json
import logging
import platform
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COLUMNAS_MEDICIONES = ["id_pozo", "fecha", "profundidad_nivel_m", "conductividad_uScm"]

ETIQUETAS = {
    "profundidad_nivel_m": "Profundidad del nivel (m)",
    "conductividad_uScm": "Conductividad eléctrica (µS/cm)",
    "cota_piezometrica_m": "Cota piezométrica (m s.n.m.)",
}

COLORES_ACUIFERO = {"Aluvial": "#2a78d6", "Carbonatado": "#eb6834", "Detritico": "#1baf7a"}


# ---------------------------------------------------------------------------
# 1. Carga
# ---------------------------------------------------------------------------
def cargar_mediciones(ruta):
    """Lee un CSV de mediciones y comprueba que tiene las columnas obligatorias.

    Parameters
    ----------
    ruta : str o Path
        Ruta al archivo CSV.

    Returns
    -------
    pandas.DataFrame
        Mediciones con la columna ``fecha`` convertida a fecha.

    Raises
    ------
    FileNotFoundError
        Si el archivo no existe.
    ValueError
        Si falta alguna columna obligatoria.
    """
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    datos = pd.read_csv(ruta, parse_dates=["fecha"])

    faltan = set(COLUMNAS_MEDICIONES) - set(datos.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(faltan)}")
    return datos


# ---------------------------------------------------------------------------
# 2. Control de calidad
# ---------------------------------------------------------------------------
def informe_calidad(datos, pozos):
    """Cuenta los problemas de calidad de unas mediciones (no modifica nada).

    Returns
    -------
    dict
        Número de filas afectadas por cada tipo de problema.
    """
    clave = ["id_pozo", "fecha"]
    exactos = datos.duplicated()
    logicos = datos.duplicated(subset=clave, keep=False) & ~datos.duplicated(keep=False)
    con_pozo = datos.merge(pozos[["id_pozo", "profundidad_pozo_m"]], on="id_pozo", how="left")
    imposibles = (con_pozo["profundidad_nivel_m"] < 0) | (
        con_pozo["profundidad_nivel_m"] > con_pozo["profundidad_pozo_m"])
    return {
        "filas": len(datos),
        "duplicados_exactos": int(exactos.sum()),
        "filas_duplicado_logico": int(logicos.sum()),
        "id_pozo_inexistente": int((~datos["id_pozo"].isin(pozos["id_pozo"])).sum()),
        "profundidad_ausente": int(datos["profundidad_nivel_m"].isna().sum()),
        "profundidad_imposible": int(imposibles.sum()),
        "ce_sospechosa_mS_cm": int((datos["conductividad_uScm"] < 50).sum()),
    }


def eliminar_duplicados_exactos(datos):
    """Devuelve una copia sin filas repetidas y el número de filas eliminadas."""
    limpios = datos.drop_duplicates().copy()
    return limpios, len(datos) - len(limpios)


def corregir_identificadores(datos, correcciones):
    """Sustituye identificadores de pozo mal escritos.

    Parameters
    ----------
    datos : pandas.DataFrame
        Mediciones con la columna ``id_pozo``.
    correcciones : dict
        Pares {"incorrecto": "correcto"}, por ejemplo {"P1": "P01"}.

    Returns
    -------
    tuple (pandas.DataFrame, int)
        Copia corregida y número de filas modificadas.
    """
    resultado = datos.copy()
    a_corregir = resultado["id_pozo"].isin(list(correcciones))
    resultado["id_pozo"] = resultado["id_pozo"].replace(correcciones)
    return resultado, int(a_corregir.sum())


def anular_duplicados_logicos(datos, variable="profundidad_nivel_m"):
    """Deja una sola fila por pozo y fecha.

    Si una misma clave pozo-fecha tenía valores distintos de ``variable``, no se
    puede saber cuál es el correcto: se conserva la fila y el valor pasa a NaN.
    Devuelve la copia y el número de claves en conflicto.
    """
    clave = ["id_pozo", "fecha"]
    resultado = datos.copy()
    valores_distintos = resultado.groupby(clave)[variable].transform("nunique")
    en_conflicto = valores_distintos > 1
    n_conflictos = len(resultado.loc[en_conflicto, clave].drop_duplicates())
    resultado.loc[en_conflicto, variable] = np.nan
    resultado = resultado.drop_duplicates(subset=clave, keep="first")
    return resultado, n_conflictos


def anular_imposibles(datos, pozos):
    """Anula (NaN) las profundidades negativas o mayores que la profundidad del pozo.

    Parameters
    ----------
    datos : pandas.DataFrame
        Mediciones con ``id_pozo`` y ``profundidad_nivel_m``.
    pozos : pandas.DataFrame
        Tabla de pozos con ``id_pozo`` y ``profundidad_pozo_m``.

    Returns
    -------
    tuple (pandas.DataFrame, int)
        Copia con los valores imposibles anulados y número de valores anulados.
    """
    resultado = datos.merge(pozos[["id_pozo", "profundidad_pozo_m"]], on="id_pozo", how="left")
    imposible = (resultado["profundidad_nivel_m"] < 0) | (
        resultado["profundidad_nivel_m"] > resultado["profundidad_pozo_m"])
    resultado.loc[imposible, "profundidad_nivel_m"] = np.nan
    resultado = resultado.drop(columns="profundidad_pozo_m")
    return resultado, int(imposible.sum())


def corregir_unidades_ce(datos, umbral_uScm=50):
    """Convierte a µS/cm las conductividades anotadas en mS/cm (valores < umbral).

    Devuelve la copia y el número de valores corregidos.
    """
    resultado = datos.copy()
    en_mS = resultado["conductividad_uScm"] < umbral_uScm
    resultado.loc[en_mS, "conductividad_uScm"] = resultado.loc[en_mS, "conductividad_uScm"] * 1000
    return resultado, int(en_mS.sum())


def limpiar_mediciones(datos, pozos, correcciones_id=None):
    """Aplica en orden las reglas de limpieza acordadas en M8_8.27.

    Returns
    -------
    tuple (pandas.DataFrame, pandas.DataFrame)
        Mediciones limpias y registro de los pasos aplicados.
    """
    registro = []

    limpios, n = eliminar_duplicados_exactos(datos)
    registro.append({"paso": "Eliminar duplicados exactos", "filas_afectadas": n})

    if correcciones_id:
        limpios, n = corregir_identificadores(limpios, correcciones_id)
        registro.append({"paso": f"Corregir identificadores {correcciones_id}", "filas_afectadas": n})

    limpios, n = anular_duplicados_logicos(limpios)
    registro.append({"paso": "Anular duplicados lógicos (valores contradictorios)", "filas_afectadas": n})

    limpios, n = anular_imposibles(limpios, pozos)
    registro.append({"paso": "Anular profundidades imposibles", "filas_afectadas": n})

    limpios, n = corregir_unidades_ce(limpios)
    registro.append({"paso": "Corregir CE anotada en mS/cm", "filas_afectadas": n})

    limpios = limpios.sort_values(["id_pozo", "fecha"]).reset_index(drop=True)
    return limpios, pd.DataFrame(registro)


# ---------------------------------------------------------------------------
# 3. Cálculo y resumen
# ---------------------------------------------------------------------------
def anadir_cota_piezometrica(datos, pozos):
    """Devuelve una copia con acuífero, cota del terreno y cota piezométrica (h = z - d)."""
    resultado = datos.merge(pozos[["id_pozo", "acuifero", "cota_terreno_m"]],
                            on="id_pozo", how="left", validate="many_to_one")
    resultado["cota_piezometrica_m"] = resultado["cota_terreno_m"] - resultado["profundidad_nivel_m"]
    return resultado


def resumir_pozo(datos_pozo, variable):
    """Devuelve un diccionario con los estadísticos básicos de una variable de un pozo."""
    serie = datos_pozo[variable]
    return {
        "n_validos": int(serie.count()),
        "n_ausentes": int(serie.isna().sum()),
        "media": serie.mean(),
        "mediana": serie.median(),
        "minimo": serie.min(),
        "maximo": serie.max(),
        "desv_tipica": serie.std(),
    }


def comprobar_resumen(resumen, pozos, variable, n_esperado=None):
    """Busca incoherencias en una tabla resumen por pozo y devuelve una lista de problemas."""
    problemas = []

    desconocidos = sorted(set(resumen.index) - set(pozos["id_pozo"]))
    if desconocidos:
        problemas.append(f"Pozos que no existen en la tabla de pozos: {desconocidos}")

    if n_esperado is not None:
        total = resumen["n_validos"] + resumen["n_ausentes"]
        exceso = list(resumen.index[total > n_esperado])
        if exceso:
            problemas.append(f"Pozos con más de {n_esperado} registros (¿duplicados?): {exceso}")

    if variable == "profundidad_nivel_m":
        prof_pozo = pozos.set_index("id_pozo")["profundidad_pozo_m"]
        mas_hondo = [p for p in resumen.index if p in prof_pozo and resumen.loc[p, "maximo"] > prof_pozo[p]]
        if mas_hondo:
            problemas.append(f"Profundidad máxima mayor que la del pozo: {mas_hondo}")

    return problemas


# ---------------------------------------------------------------------------
# 4. Salidas
# ---------------------------------------------------------------------------
def guardar_figura_serie(datos_pozo, variable, ruta, dpi=150):
    """Dibuja la serie temporal de ``variable`` para un pozo y la guarda en ``ruta``.

    Returns
    -------
    Path
        Ruta del archivo creado.
    """
    if datos_pozo.empty:
        raise ValueError("No hay datos para dibujar.")

    datos_pozo = datos_pozo.sort_values("fecha")
    id_pozo = datos_pozo["id_pozo"].iloc[0]
    acuifero = datos_pozo["acuifero"].iloc[0] if "acuifero" in datos_pozo else None

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(datos_pozo["fecha"], datos_pozo[variable], linewidth=1.6, marker="o", markersize=3,
            color=COLORES_ACUIFERO.get(acuifero, "#52514e"))
    if variable == "profundidad_nivel_m":
        ax.invert_yaxis()
    ax.set_title(f"{id_pozo} ({acuifero}) · {ETIQUETAS.get(variable, variable)}", loc="left")
    ax.set_xlabel("Fecha")
    ax.set_ylabel(ETIQUETAS.get(variable, variable))
    ax.grid(alpha=0.3)
    fig.tight_layout()

    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(ruta, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return ruta


def crear_registrador(ruta_log):
    """Crea un registrador (logger): todos los mensajes van a ``ruta_log``; en pantalla, solo avisos y errores."""
    registrador = logging.getLogger("flujo_m8")
    registrador.setLevel(logging.INFO)
    for manejador in list(registrador.handlers):     # evita mensajes duplicados si se ejecuta dos veces
        manejador.close()
        registrador.removeHandler(manejador)
    formato = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")
    archivo = logging.FileHandler(ruta_log, mode="w", encoding="utf-8")   # todo (INFO y superior) al archivo
    pantalla = logging.StreamHandler(sys.stdout)
    pantalla.setLevel(logging.WARNING)                                    # en pantalla, solo avisos y errores
    for manejador in [archivo, pantalla]:
        manejador.setFormatter(formato)
        registrador.addHandler(manejador)
    return registrador


def cerrar_registrador(registrador):
    """Cierra los archivos abiertos por el registrador."""
    for manejador in list(registrador.handlers):
        manejador.close()
        registrador.removeHandler(manejador)


# ---------------------------------------------------------------------------
# 5. Flujo completo
# ---------------------------------------------------------------------------
def ejecutar_flujo(config, raiz, mediciones_extra=None):
    """Ejecuta el flujo completo: cargar → validar → limpiar → calcular → resumir → figuras → registro.

    Parameters
    ----------
    config : dict
        Configuración (ver config.json).
    raiz : str o Path
        Carpeta raíz del repositorio; las rutas de ``config`` son relativas a ella.
    mediciones_extra : pandas.DataFrame, opcional
        Mediciones nuevas, ya armonizadas, que se añaden a las históricas.

    Returns
    -------
    pandas.DataFrame
        Tabla resumen por pozo.
    """
    raiz = Path(raiz)
    salida = raiz / config["carpeta_salida"]
    for sub in ["tablas", "figuras", "logs"]:
        (salida / sub).mkdir(parents=True, exist_ok=True)
    log = crear_registrador(salida / "logs" / "ejecucion.log")
    log.info("Inicio del flujo")

    pozos = pd.read_csv(raiz / config["archivo_pozos"])
    mediciones = cargar_mediciones(raiz / config["archivo_mediciones"])
    log.info("Mediciones cargadas: %d filas", len(mediciones))
    if mediciones_extra is not None:
        mediciones = pd.concat([mediciones, mediciones_extra[COLUMNAS_MEDICIONES]], ignore_index=True)
        log.info("Añadidas %d mediciones nuevas", len(mediciones_extra))

    calidad = informe_calidad(mediciones, pozos)
    for problema, n in calidad.items():
        if problema != "filas" and n > 0:
            log.warning("Calidad · %s: %d", problema, n)

    limpios, pasos = limpiar_mediciones(mediciones, pozos, config.get("correcciones_id"))
    datos = anadir_cota_piezometrica(limpios, pozos)
    limpios.to_csv(salida / "tablas" / "mediciones_limpias.csv", index=False)
    pasos.to_csv(salida / "tablas" / "pasos_limpieza.csv", index=False)

    lista_pozos = sorted(datos["id_pozo"].unique()) if config["pozos"] == "todos" else config["pozos"]
    variable = config["variable"]
    archivos = []
    filas_resumen = {}
    for id_pozo in lista_pozos:
        try:
            datos_pozo = datos.loc[datos["id_pozo"] == id_pozo]
            if datos_pozo.empty:
                raise ValueError(f"el pozo {id_pozo} no tiene datos")
            filas_resumen[id_pozo] = resumir_pozo(datos_pozo, variable)
            ruta_fig = salida / "figuras" / f"{id_pozo}_{variable}.{config['formato_figura']}"
            archivos.append(guardar_figura_serie(datos_pozo, variable, ruta_fig, dpi=config["dpi"]))
            log.info("Procesado %s", id_pozo)
        except ValueError as error:
            log.error("No se pudo procesar %s: %s", id_pozo, error)

    resumen = pd.DataFrame(filas_resumen).T
    resumen.index.name = "id_pozo"
    ruta_resumen = salida / "tablas" / f"resumen_{variable}.csv"
    resumen.round(3).to_csv(ruta_resumen)
    archivos += [ruta_resumen, salida / "tablas" / "mediciones_limpias.csv", salida / "tablas" / "pasos_limpieza.csv"]

    problemas = comprobar_resumen(resumen, pozos, variable, n_esperado=config.get("registros_por_pozo"))
    for p in problemas:
        log.error("Comprobación final · %s", p)

    registro = {
        "fecha_ejecucion_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "sistema": platform.system(),
        "versiones": {"numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__},
        "configuracion": config,
        "calidad_datos_entrada": calidad,
        "pasos_limpieza": pasos.to_dict(orient="records"),
        "problemas_comprobacion_final": problemas,
        "archivos_generados": [
            {"archivo": str(Path(a).relative_to(raiz)), "bytes": Path(a).stat().st_size} for a in archivos
        ],
    }
    (salida / "logs" / "registro_ejecucion.json").write_text(
        json.dumps(registro, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    log.info("Flujo terminado: %d pozos, %d archivos", len(resumen), len(archivos))
    cerrar_registrador(log)
    return resumen

Ahora **importamos** el módulo. Como la carpeta `mi_proyecto` está dentro de la raíz del repositorio (que es la carpeta de trabajo), basta con:

In [ ]:
from mi_proyecto import hidro

help(hidro.limpiar_mediciones)

> ⚠️ **Error frecuente al editar un módulo.** Python importa cada módulo **una sola vez** por sesión. Si modificas `hidro.py` y vuelves a ejecutar `from mi_proyecto import hidro`, **seguirás usando la versión antigua**. Hay que recargarlo explícitamente con `importlib.reload(hidro)`, o reiniciar la sesión.

In [ ]:
hidro = importlib.reload(hidro)   # ejecuta esto cada vez que modifiques hidro.py

El módulo contiene la limpieza completa de M8_8.27 en una sola llamada, que además devuelve el **registro** de lo que ha hecho:

In [ ]:
limpios, pasos = hidro.limpiar_mediciones(mediciones, pozos, correcciones_id={"P1": "P01"})
pasos

> 📖 **Documentación de tu propio código.** Escribe `hidro.` en una celda y pulsa **Tab**: verás la lista de funciones del módulo. Sitúa el cursor dentro de los paréntesis de `hidro.resumir_pozo(` y aparecerá su docstring. Tu módulo se documenta igual que pandas o NumPy.

---
# M8_8.32 · Automatización y ejecución reproducible

## 8. Automatizar no es solo repetir

Automatizar significa aplicar un procedimiento **ya comprendido y probado** a muchas unidades (pozos, archivos, escenarios) de forma **consistente**, **registrando** lo que ocurre y generando resultados **trazables**.

> 💡 **Automatizar un procedimiento incorrecto solo repite el error más deprisa.** Por eso primero hemos comprendido los datos (M8_8.23–27) y después hemos escrito y probado las funciones (M8_8.31).

## 9. Encuentra el fallo: automatizar sin controles

Supongamos que alguien automatiza un resumen de la profundidad por pozo, pero **sin limpiar los datos**. El código funciona y no da ningún error:

In [ ]:
crudos = hidro.cargar_mediciones(INPUT / "mediciones.csv")

resumen_crudo = pd.DataFrame({
    id_pozo: hidro.resumir_pozo(grupo, "profundidad_nivel_m")
    for id_pozo, grupo in crudos.groupby("id_pozo")
}).T
resumen_crudo.round(2)

¿Ves los problemas? Mira con atención las columnas `n_validos` y `maximo`, y la lista de pozos. Una función de **comprobación** los detecta automáticamente:

In [ ]:
for problema in hidro.comprobar_resumen(resumen_crudo, pozos, "profundidad_nivel_m", n_esperado=60):
    print("⚠", problema)

In [ ]:
datos = hidro.anadir_cota_piezometrica(limpios, pozos)
resumen_limpio = pd.DataFrame({
    id_pozo: hidro.resumir_pozo(grupo, "profundidad_nivel_m")
    for id_pozo, grupo in datos.groupby("id_pozo")
}).T

problemas = hidro.comprobar_resumen(resumen_limpio, pozos, "profundidad_nivel_m", n_esperado=60)
print("Problemas tras la limpieza:", problemas if problemas else "ninguno ✔")

> 💡 **Un flujo automático debe incluir sus propias comprobaciones**, y estas deben poder **fallar**. Si una comprobación no puede fallar nunca, no está comprobando nada.

## 10. Configuración separada de la lógica

Las decisiones que pueden cambiar entre ejecuciones (qué pozos, qué variable, qué correcciones, formato de las figuras…) no deberían estar escritas dentro de las funciones. Se reúnen en un **archivo de configuración**, que también documenta con qué parámetros se obtuvo cada resultado.

**JSON** es un formato de texto legible, parecido a un diccionario de Python. Las rutas se escriben **relativas a la raíz** del repositorio.

In [ ]:
configuracion = {
    "archivo_pozos": "data/input/pozos.csv",
    "archivo_mediciones": "data/input/mediciones.csv",
    "carpeta_salida": "mi_proyecto/resultados",
    "pozos": "todos",
    "variable": "profundidad_nivel_m",
    "correcciones_id": {"P1": "P01"},
    "registros_por_pozo": 60,
    "formato_figura": "png",
    "dpi": 150,
}

ruta_config = ROOT / "mi_proyecto" / "config.json"
ruta_config.write_text(json.dumps(configuracion, indent=2, ensure_ascii=False), encoding="utf-8")
print(ruta_config.read_text())

## 11. El flujo completo

`hidro.ejecutar_flujo` (léela en el módulo) coordina todas las funciones:

```text
cargar → informe de calidad → limpiar → cota piezométrica → para cada pozo: resumir + figura
       → tabla resumen → comprobaciones finales → registro de ejecución (JSON) + log
```

Aspectos que hacen el flujo **robusto y trazable**:

- **Nombres de archivo trazables:** `P03_profundidad_nivel_m.png` dice qué pozo y qué variable contiene.
- **Errores esperados con `try/except`:** si un pozo no tiene datos, se registra el error y se continúa con el siguiente, en lugar de detener todo el flujo. Solo se capturan los errores que **sabemos gestionar** (`ValueError`), nunca todos a ciegas.
- **Registro (*log*):** mensajes `INFO` (todo va bien), `WARNING` (incidencia que permite seguir) y `ERROR` (algo no se pudo hacer), con fecha y hora, en pantalla y en un archivo.
- **Registro de ejecución:** un JSON con la fecha, las versiones, la configuración, el informe de calidad, los pasos de limpieza y la lista de archivos generados.

In [ ]:
resumen = hidro.ejecutar_flujo(configuracion, ROOT)
resumen.round(2)

En pantalla solo aparecen los avisos (`WARNING`) y errores. El archivo de registro guarda **todos** los mensajes, con su hora:

In [ ]:
print((ROOT / configuracion["carpeta_salida"] / "logs" / "ejecucion.log").read_text())

In [ ]:
salida = ROOT / configuracion["carpeta_salida"]
for archivo in sorted(salida.rglob("*")):
    if archivo.is_file():
        print(f"{archivo.relative_to(ROOT)}  ({archivo.stat().st_size / 1024:.1f} kB)")

In [ ]:
print((salida / "logs" / "registro_ejecucion.json").read_text()[:1500], "...")

Una de las figuras generadas:

In [ ]:
from IPython.display import Image
Image(filename=str(salida / "figuras" / "P03_profundidad_nivel_m.png"))

## 12. Idempotencia

Un proceso es **idempotente** si ejecutarlo dos veces con las mismas entradas deja **el mismo resultado** que ejecutarlo una vez: no duplica filas, no acumula archivos con nombres distintos y no depende de lo que quedó de la ejecución anterior.

Nuestro flujo lo es porque sobrescribe siempre los mismos archivos (`mode="w"` en el log, nombres fijos para las salidas) y no modifica nunca los datos de entrada. Compruébalo:

In [ ]:
resumen_2 = hidro.ejecutar_flujo(configuracion, ROOT)
print("¿Mismo resultado en la segunda ejecución?", resumen.equals(resumen_2))

## 13. De notebook a script

Un **script** ejecuta el flujo completo sin abrir ningún notebook: por ejemplo, cada noche en un servidor o cuando llegan datos nuevos. El patrón `if __name__ == "__main__":` hace que la función `main()` se ejecute **solo** cuando el archivo se lanza como script, y no cuando se importa como módulo.

In [ ]:
%%writefile mi_proyecto/ejecutar_flujo.py
"""Ejecuta el flujo de análisis del proyecto.

Uso (desde la raíz del repositorio):
    python mi_proyecto/ejecutar_flujo.py
"""
import json
from pathlib import Path

from hidro import ejecutar_flujo

CARPETA_PROYECTO = Path(__file__).resolve().parent
RAIZ = CARPETA_PROYECTO.parent


def main():
    configuracion = json.loads((CARPETA_PROYECTO / "config.json").read_text(encoding="utf-8"))
    resumen = ejecutar_flujo(configuracion, RAIZ)
    print(resumen.round(2).to_string())


if __name__ == "__main__":
    main()

Para ejecutar un script se usa la **terminal**. En un notebook, el signo `!` al principio de una línea envía esa orden a la terminal del sistema. Así se ve exactamente lo que ocurriría en una terminal:

In [ ]:
!python mi_proyecto/ejecutar_flujo.py

### ✏️ Ejercicio 2 · Otra variable, otros pozos

Sin tocar `hidro.py`, cambia **solo la configuración** para analizar la **conductividad eléctrica** (`conductividad_uScm`) de los pozos del **acuífero aluvial**, y guarda los resultados en `mi_proyecto/resultados_ce_aluvial`.

1. Crea `config_ce` como una copia de `configuracion` con `dict(configuracion)` y modifica las claves necesarias.
2. Obtén la lista de pozos aluviales a partir de la tabla `pozos`.
3. Ejecuta el flujo y guarda el resumen en `resumen_ce`.

In [ ]:
config_ce = dict(configuracion)
config_ce["variable"] = ...
config_ce["pozos"] = ...
config_ce["carpeta_salida"] = ...

resumen_ce = hidro.ejecutar_flujo(config_ce, ROOT)
resumen_ce.round(1)

In [ ]:
# ✅ Comprobación
carpeta_figuras = ROOT / "mi_proyecto" / "resultados_ce_aluvial" / "figuras"
assert sorted(resumen_ce.index) == ["P01", "P02", "P03", "P09", "P10"]
assert (carpeta_figuras / "P03_conductividad_uScm.png").exists(), "Revisa el nombre de la variable y la carpeta de salida"
assert resumen_ce.loc["P09", "minimo"] > 500, "La conductividad en mS/cm de P09 debería estar corregida"
print("✔ Correcto. Con solo cambiar la configuración, el mismo código produce un análisis diferente.")

---
# M8_8.33 · Taller: un proyecto reproducible con datos nuevos

## 14. El encargo

> La comunidad de usuarios del acuífero ha enviado las medidas del **primer trimestre de 2026** (`data/input/campana_2026_T1.csv`). Quiere saber **si los niveles se han recuperado tras la sequía**, y necesita un análisis que el equipo técnico pueda **repetir cada trimestre** sin tu ayuda.

Tu entregable es la carpeta `mi_proyecto/` con:

1. una función que lea y **armonice** el archivo de campo;
2. el flujo ejecutado con los datos históricos **y** los nuevos;
3. la configuración y el registro de ejecución;
4. un **README** que explique cómo repetir el análisis;
5. una **interpretación** breve con una limitación.

## 15. Paso 1 · Leer y armonizar los datos nuevos

En M8_8.23 aprendiste a leer este archivo. Ahora conviértelo en una **función** que devuelva un DataFrame con **las mismas columnas y tipos** que las mediciones históricas (`hidro.COLUMNAS_MEDICIONES`). Así el resto del flujo no necesita saber de dónde vienen los datos.

In [ ]:
print((INPUT / "campana_2026_T1.csv").read_text()[:200])

### ✏️ Ejercicio 3 · `leer_campana_campo`

La función debe:

1. leer el archivo con el separador, el decimal y el código de ausentes correctos;
2. **renombrar** las columnas a `id_pozo`, `fecha`, `profundidad_nivel_m` y `conductividad_uScm` (📖 `DataFrame.rename`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html);
3. convertir la fecha con el formato día/mes/año;
4. devolver solo las columnas de `hidro.COLUMNAS_MEDICIONES`, en ese orden.

In [ ]:
def leer_campana_campo(ruta):
    """..."""
    ...
    return ...


campana = leer_campana_campo(INPUT / "campana_2026_T1.csv")
campana.head()

In [ ]:
# ✅ Comprobación
assert list(campana.columns) == hidro.COLUMNAS_MEDICIONES
assert campana.shape == (36, 4)
assert pd.api.types.is_datetime64_any_dtype(campana["fecha"]) and campana["fecha"].max() == pd.Timestamp("2026-03-01")
assert campana["profundidad_nivel_m"].isna().sum() == 1 and campana["profundidad_nivel_m"].min() > 0
print("✔ Correcto. Los datos nuevos tienen el mismo formato que los históricos.")

## 16. Paso 2 · Ejecutar el flujo con los datos nuevos

Actualiza la configuración. Ahora cada pozo tiene **63** registros (60 históricos + 3 nuevos), y la configuración debe reflejarlo. Si no, la comprobación final avisará con razón: **la configuración documenta lo que esperamos de los datos**.

In [ ]:
configuracion["carpeta_salida"] = "mi_proyecto/resultados_2026T1"
configuracion["registros_por_pozo"] = 63
configuracion["archivo_campana"] = "data/input/campana_2026_T1.csv"   # documentamos el origen de los datos nuevos

ruta_config.write_text(json.dumps(configuracion, indent=2, ensure_ascii=False), encoding="utf-8")

resumen_2026 = hidro.ejecutar_flujo(configuracion, ROOT, mediciones_extra=campana)
resumen_2026.round(2)

## 17. Paso 3 · Responder a la pregunta

¿Se han recuperado los niveles? Comparamos la profundidad media del **primer trimestre** de cada año, porque comparar meses distintos mezclaría la variación estacional con la recuperación.

In [ ]:
todas = pd.read_csv(ROOT / configuracion["carpeta_salida"] / "tablas" / "mediciones_limpias.csv", parse_dates=["fecha"])
todas = hidro.anadir_cota_piezometrica(todas, pozos)

primer_trimestre = todas.loc[todas["fecha"].dt.month.isin([1, 2, 3])].copy()
primer_trimestre["año"] = primer_trimestre["fecha"].dt.year
tabla_t1 = (primer_trimestre
            .groupby(["acuifero", "año"])["profundidad_nivel_m"].mean()
            .unstack("año").round(2))
tabla_t1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for acuifero, fila in tabla_t1.iterrows():
    ax.plot(fila.index, fila.values, marker="o", linewidth=2, color=hidro.COLORES_ACUIFERO[acuifero], label=acuifero)
    ax.text(fila.index[-1] + 0.1, fila.values[-1], acuifero, va="center", fontsize=9)
ax.invert_yaxis()
ax.set_xticks(tabla_t1.columns)
ax.set_xlabel("Año (primer trimestre)")
ax.set_ylabel("Profundidad media del nivel (m)")
ax.set_title("Profundidad media en enero–marzo de cada año")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig(ROOT / configuracion["carpeta_salida"] / "figuras" / "comparacion_primer_trimestre.png", dpi=150, bbox_inches="tight")
plt.show()

## 18. Paso 4 · El README del proyecto

Un proyecto sin instrucciones no es reproducible. Completa la plantilla (edita el texto entre corchetes) y ejecuta la celda para guardarla.

In [ ]:
%%writefile mi_proyecto/README.md
# Seguimiento trimestral de niveles · [tu nombre]

## Pregunta
[¿Qué pregunta responde este análisis?]

## Datos
- Históricos: `data/input/mediciones.csv` y `data/input/pozos.csv` (2021–2025).
- Nuevos: [archivo y periodo].
- Problemas de calidad encontrados y cómo se han tratado: ver `resultados_2026T1/tablas/pasos_limpieza.csv`.

## Cómo repetir el análisis
1. [Paso 1]
2. [Paso 2]

## Resultados
- [Tabla o figura principal y dónde está]

## Limitaciones
- [Al menos una]

## 19. Paso 5 · Interpretación

Escribe en esta celda **3–5 frases**: qué muestran los resultados, cómo responden a la pregunta y **una limitación** del análisis. Distingue lo que **observas** de lo que **interpretas**.

*[Tu interpretación aquí]*

<details>
<summary>Ejemplo de interpretación (despliégalo después de escribir la tuya)</summary>

En el primer trimestre de 2026, la profundidad media del nivel es la menor de toda la serie en los tres acuíferos (unos 6.7 m en el aluvial, 21.8 m en el carbonatado y 14.0 m en el detrítico), por lo que los niveles se han recuperado de la sequía de 2022–2024. La recuperación ha sido distinta en cada acuífero: el aluvial y el detrítico, que responden en uno o dos meses a la lluvia, ya mejoraron en 2025, mientras que en el carbonatado, de respuesta lenta, el nivel de enero–marzo de 2025 todavía estaba más de 1 m por debajo del de 2021 y la recuperación no se completa hasta 2026. Limitación: solo comparamos tres meses de un periodo húmedo y no consideramos los bombeos; para afirmar que las reservas se han repuesto habría que seguir la evolución durante un ciclo hidrológico completo.
</details>

## 20. Paso 6 · Comprobación final: ¿es realmente reproducible?

- [ ] *Entorno de ejecución → Reiniciar sesión y ejecutar todo* funciona sin errores.
- [ ] Los datos de `data/input` no se han modificado.
- [ ] Todas las rutas son relativas a la raíz del repositorio.
- [ ] `config.json` contiene todos los parámetros del análisis.
- [ ] `ejecucion.log` no contiene mensajes `ERROR`, o están explicados.
- [ ] `registro_ejecucion.json` incluye versiones, configuración y archivos generados.
- [ ] El README permite a otra persona repetir el análisis. **Pruébalo:** pide a un compañero que abra tu notebook y lo ejecute.
- [ ] La interpretación incluye al menos una limitación.

> ⚠️ **Recuerda:** en Colab, `mi_proyecto/` se borra al cerrar la sesión. Descarga la carpeta antes de salir. La celda siguiente la comprime en un único archivo `.zip` que puedes descargar desde el panel de archivos de la izquierda.

In [ ]:
import shutil

ruta_zip = shutil.make_archive(str(ROOT / "mi_proyecto_entrega"), "zip", root_dir=ROOT, base_dir="mi_proyecto")
print("Archivo para descargar:", ruta_zip)

## Síntesis de la tarde

- **Funciones:** una responsabilidad, parámetros en lugar de valores fijos, `return`, sin modificar las entradas, docstring y pruebas con `assert`.
- **Módulos y scripts:** las funciones se guardan en un `.py` y se importan. Tras editar un módulo, hay que recargarlo con `importlib.reload`. `if __name__ == "__main__"` distingue ejecutar de importar.
- **Automatización:** configuración separada del código, bucles sobre pozos, nombres trazables, `try/except` solo para errores conocidos, *logging*, comprobaciones que pueden fallar e idempotencia.
- **Reproducibilidad:** datos originales intactos, rutas relativas, configuración, registro de versiones, README y ejecución desde cero.
- **Juicio científico:** el flujo automatiza las **decisiones que ya tomamos**, pero no las toma por nosotros. La elección de las reglas de limpieza, la comparación adecuada (mismo trimestre) y la interpretación siguen siendo trabajo del hidrogeólogo.

### Conexión con los bloques siguientes

- **M8_8.34–36:** usaréis la **API** de un código de modelización numérica: funciones, módulos, arrays y documentación, exactamente como aquí.
- **M8_8.37–42:** el *machine learning* necesita datos limpios, divididos correctamente (recuerda la dependencia temporal) y flujos reproducibles.